# From Literature to Benchmarks: 

candidates are the papers most similar to the hand-reviewed papers.

LLM API pipeline (heavy-ion jets)

The notebook continues hackaton notebook, found the task clusters on the CPU, and it
defined 4 downstream tasks by hand in downstream_tasks.json. 

This notebook searches the **whole heavy-ion pool** for more papers on those tasks, using the LLM API. The saved cell outputs and `final_result.json` document the original `nb04-v1` run. The updated prompts below use the prediction objective to match studies. A fresh run needs cached responses for those prompts or API access.

## 1. Setup
**Goal:** load the libraries and the API client, and set where files are read from and written to.

In [ ]:
# Libraries. The API client, the text cleaner and the quote checker come from lit2bench-preparation/api_tests.
import os, re, sys, json, subprocess
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd

import ilaas_client
from ilaas_client import ILaaSClient, parse_json_answer
from textutils import strip_latex, quote_match

pd.set_option("display.max_colwidth", 120)

In [3]:
# Inputs and the folder for this notebook's results.
SRC = "/lustre/scratch/data/jdearrud_hpc-inspire/release_20261104"        
NB03_DIR = "baseline_outputs_notebook_03"
POOL_FILE = os.path.join(NB03_DIR, "pbpb_papers.jsonl")                
REFERENCE_FILE = os.path.join(NB03_DIR, "downstream_tasks_cluster_1.json")  #hand reviewd reference

OUTPUT_DIR = "api_outputs_notebook_04_seed_only"
os.makedirs(OUTPUT_DIR, exist_ok=True)


def out_path(name):
    """Full path of a file inside OUTPUT_DIR."""
    return os.path.join(OUTPUT_DIR, name)


# The API log and the answer cache go into OUTPUT_DIR.
ilaas_client.LOG_DIR = Path(out_path("api_logs"))
ilaas_client.LOG_FILE = ilaas_client.LOG_DIR / "api_calls.jsonl"
ilaas_client.CACHE_DIR = ilaas_client.LOG_DIR / "cache"
ilaas_client.LOG_DIR.mkdir(exist_ok=True)

In [ ]:
MODEL, RERANK, EMBED = "qwen-3.6-35b-instruct", "bge-reranker-v2-m3", "bge-m3"
WORKERS = 4                     # at most 4 parallel requests per team
CAND_MIN, CAND_MAX = 50, 200    # Block 2: candidates per task
QUEUE_MIN, QUEUE_MAX = 20, 30   # Block 3: papers per task that go on to verification
ANCHOR_SHARE = 0.02             # a task term hitting > 2 % of the pool is too generic to make a candidate on its own
CHUNK, PASSAGES = 1500, 3       # Block 4: full-text chunk size (characters) and passages sent to the LLM
SAME_STUDY, CHECK_STUDY = 0.95, 0.90   # bge-m3 cosine: same study / check by hand
FIELD = "heavy-ion"
PROMPT_VERSION = "nb04-v2-output"      # change it when you edit a prompt; it is written into retrieval_method

In [5]:
KEY_FILE = os.path.expanduser("~/.ilaas_api_key")
if not os.environ.get("ILAAS_API_KEY") and os.path.exists(KEY_FILE):
    os.environ["ILAAS_API_KEY"] = open(KEY_FILE).read().strip()

client = ILaaSClient(tag="nb04_heavy_ion", use_cache=True, verbose=False)
if client.has_key:
    print("API reachable, models:", client.models())
else:
    print("No API key: only cached answers can be replayed (new calls fail with 'cache miss').")

API reachable, models: ['gemma-4-31b', 'qwen-3.6-35b-instruct-reasoning-high', 'qwen-3.8-27b', 'llama-3.1-8b', 'qwen-3.6-35b-instruct', 'bge-reranker-v2-m3', 'bge-m3']


In [6]:
FAILED_CALLS = []      # calls that failed after all retries (or cache misses offline); their papers become uncertain


def as_text(v):
    """Models sometimes return a list or a number where a string was asked for."""
    if v is None:
        return ""
    if isinstance(v, list):
        return " ".join(as_text(x) for x in v)
    return v if isinstance(v, str) else str(v)


def chat_json(system, user, max_tokens=1500):
    """One JSON answer. A failed call returns {} (-> decision 'invalid' -> uncertain) instead of stopping the batch."""
    msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": user}]
    try:
        r = client.chat(MODEL, msgs, response_format={"type": "json_object"}, temperature=0,
                        max_tokens=max_tokens, thinking=False)
    except Exception as e:
        FAILED_CALLS.append(f"{type(e).__name__}: {str(e)[:200]}")
        return {}
    ans = parse_json_answer(r["choices"][0]["message"].get("content") or "") or {}
    return {k: (v if isinstance(v, list) and k == "other_tasks" else as_text(v)) for k, v in ans.items()}


def pmap(fn, items):
    """fn over items with <= WORKERS parallel API calls; results in input order."""
    with ThreadPoolExecutor(WORKERS) as ex:
        return list(ex.map(fn, items))


def rerank_scores(query, docs, batch=200):
    """Reranker relevance of every doc for the query."""
    parts = [docs[i:i + batch] for i in range(0, len(docs), batch)]
    res = pmap(lambda part: client.rerank(RERANK, query, part, top_n=len(part)), parts)
    out = []
    for part, r in zip(parts, res):
        s = [0.0] * len(part)
        for x in r["results"]:
            s[x["index"]] = x["relevance_score"]
        out += s
    return out

## 2. Load the heavy-ion pool
**Goal:** load all heavy-ion papers that notebook 03 found, and clean their text.

In [7]:
ONLY_AI = False

pool_df = pd.read_json(POOL_FILE, lines=True, dtype={"recid": str})
pool_df = pool_df[pool_df["abstract"].notna() & (pool_df["abstract"].str.strip() != "")]
pool_df = pool_df.drop_duplicates(subset="recid").reset_index(drop=True)
if ONLY_AI:
    ai_ids = set(pd.read_json(os.path.join(NB03_DIR, "papers_ai_only.jsonl"), lines=True, dtype={"recid": str})["recid"])
    pool_df = pool_df[pool_df["recid"].isin(ai_ids)].reset_index(drop=True)

# strip_latex removes LaTeX, MathML and HTML (tested on the corpus); the search uses a lower-case, dash-free copy.
pool_df["title"] = pool_df["title"].fillna("").map(strip_latex)
pool_df["abstract"] = pool_df["abstract"].map(strip_latex)
pool_df["year"] = pool_df["year"].astype(str).str[:4]


def norm(s):
    return re.sub(r"\s+", " ", re.sub(r"[-‐–/_]", " ", s.lower())).strip()


pool_df["_t"] = pool_df["title"].map(norm)
pool_df["_a"] = pool_df["abstract"].map(norm)
by_id = pool_df.set_index("recid", drop=False)
print(f"pool: {len(pool_df)} heavy-ion papers with abstract | per decade:",
      pool_df["year"].str[:3].add("0s").value_counts().to_dict())

pool: 25200 heavy-ion papers with abstract | per decade: {'2010s': 14952, '2020s': 10248}


## 3. Define the tasks
Define each task by the scientific field and prediction objective. The first notebook supplies a stable `task_id`, label and type. Different inputs or model architectures can produce the same scientific output.

The prompt lists observed inputs from the reviewed studies as examples. It asks what each new study actually uses and predicts. Keep its evaluation metric and reported result in separate fields.

Not from the JSON, hand-written:
- LOOK_ALIKES: one sentence per task naming what does **not** count. In the lit2bench tests, naming look-alikes was
  the only thing that helped against homonyms. Set LOOK_ALIKES = {} to send only the JSON content.
- SHORT_QUERY (for the reranker) and BASELINE_TERMS (keyword search). They are search helpers and never reach the
  LLM's screening or verification prompts.

In [8]:
BRACKETS = re.compile(r"\s*\([^()]*\)")   


def phrases(papers, key):
    """The papers' values for one field, without bracketed setting details, dropped phrases and duplicates."""
    out = []
    for p in papers:
        s = BRACKETS.sub("", p.get(key) or "").strip(" ;,.")
        if s and s.lower() not in {x.lower() for x in out}:
            out.append(s)
    return out


# Hand-written, not in the JSON: what does NOT count as the task.
LOOK_ALIKES = {
    "quenched_vs_unquenched_jet_classification": "measurements of jet suppression (R_AA) or of average substructure "
        "modifications that never label individual jets",
    "jet_pt_background_correction": "background subtraction for non-jet observables (flow harmonics, particle spectra)",
    "jet_energy_loss_prediction": "theory calculations or global fits of the average parton energy loss or of "
        "transport coefficients (q-hat) that do not predict it for individual jets",
    "jet_particle_vs_background_classification": "corrections of the total jet momentum without per-particle labels",
}
# Search helpers, never shown to the LLM.
# The reranker model reads each paper and scores how well it matches this sentence. The LLM never sees it.
SHORT_QUERY = {
    "quenched_vs_unquenched_jet_classification": "classification of individual quenched vs. vacuum-like jets in heavy-ion collisions",
    "jet_pt_background_correction": "jet momentum background subtraction and underlying event correction in heavy-ion collisions",
    "jet_energy_loss_prediction": "prediction of the energy loss of individual jets in the quark-gluon plasma from jet substructure",
    "jet_particle_vs_background_classification": "particle-level separation of jet particles from the underlying event background in heavy-ion collisions",
}
BASELINE_TERMS = {
    "quenched_vs_unquenched_jet_classification": ["quenched jets", "quenched jet", "vacuum like jets", "medium modified jets",
                                                  "jet quenching identification", "identify quenched jets", "tagging quenched jets"],
    "jet_pt_background_correction": ["jet momentum reconstruction", "jet background subtraction", "background subtraction for jets",
                                     "constituent subtraction", "underlying event subtraction", "background fluctuations"],
    "jet_energy_loss_prediction": ["jet energy loss", "energy loss of jets", "energy loss ratio", "jet by jet"],
    "jet_particle_vs_background_classification": ["jet particles", "background particles", "jet characterization", "particle by particle"],
}

reference = json.load(open(REFERENCE_FILE))
TASKS = {}
for t in reference["tasks"]:
    papers = [p for p in t["papers"] if p.get("decision") in ("supported", "candidate")]
    inputs, targets = phrases(papers, "input_modality"), phrases(papers, "observable_or_target")
    if not targets:
        print(f"! {t['task_id']}: no prediction target in the JSON -> skipped")
        continue
    TASKS[t["task_id"]] = {
        "task_label": t["task_label"],
        "task_type": t["task_type"],
        "input_examples": "; ".join(inputs) or "none recorded",
        "observable_or_target": "; ".join(targets),
        "description": f"{t['task_label']} ({t['task_type']})."
                       + (f" NOT this task: {LOOK_ALIKES[t['task_id']]}." if t["task_id"] in LOOK_ALIKES else ""),
        "short_query": SHORT_QUERY.get(t["task_id"], t["task_label"]),
        "baseline_terms": BASELINE_TERMS.get(t["task_id"], []),
    }

# Your hand-reviewed papers: reference for recall checks, seeds for "seed and expand", merged at the end.
ref_df = pd.DataFrame([{"task_id": t["task_id"], **p} for t in reference["tasks"] for p in t["papers"]])
ref_df["recid"] = ref_df["paper_id"].str.replace("INSPIRE:", "", regex=False)
print("hand-reviewed papers per task:", ref_df["task_id"].value_counts().to_dict(), "\n")

# What the LLM will read for each task:
for task_id, task in TASKS.items():
    print(f"== {task_id}\n  description: {task['description']}\n  example inputs: {task['input_examples']}\n"
          f"  target:      {task['observable_or_target']}\n")

hand-reviewed papers per task: {'quenched_vs_unquenched_jet_classification': 4, 'jet_energy_loss_prediction': 2, 'jet_pt_background_correction': 1, 'jet_particle_vs_background_classification': 1} 

== jet_pt_background_correction
  description: Correct jet transverse momentum for the heavy-ion underlying background (regression). NOT this task: background subtraction for non-jet observables (flow harmonics, particle spectra).
  input:       any of: several jet parameters, including properties of the jet constituents
  target:      jet transverse momentum corrected for the underlying background

== quenched_vs_unquenched_jet_classification
  description: Classify each jet as quenched (medium-modified) or unquenched (vacuum-like) (classification). NOT this task: measurements of jet suppression (R_AA) or of average substructure modifications that never label individual jets.
  input:       any of: jet declustering history tree; 4-momenta of all jet constituents; representations of individu

## 4. Candidates from found papers
**Goal:** for every task, find the papers in the pool that are most similar to the hand-reviewed papers.

1. Every jet paper in the pool (title + abstract) is turned into a vector with the bge-m3 embedding model.
   The vectors are saved to emb_bge-m3_scope.npz, so this runs only once.
2. The  hand-reviewed papers of a task are the **seeds**. Each pool paper gets its similarity to the closest seed.
3. The N_NEIGHBOURS most similar papers + the seeds themselves become the candidates.
4. The reranker scores them against short_query.


In [ ]:
N_NEIGHBOURS = 100                              # similar papers per task that go to LLM screening
RERANK_PREFILTER = re.compile(r"\bjets?\b")     # only jet papers are compared 
EMB_FILE = out_path("emb_bge-m3_scope.npz")
docs = {r: f"{t}. {a[:2000]}" for r, t, a in zip(pool_df["recid"], pool_df["title"], pool_df["abstract"])}

# 1. embed the jet papers of the pool (+ your seeds), once
jet = pool_df["_t"].str.contains(RERANK_PREFILTER) | pool_df["_a"].str.contains(RERANK_PREFILTER)
scope_ids = sorted(set(pool_df["recid"][jet]) | (set(ref_df["recid"]) & set(by_id.index)))
emb = {}
if os.path.exists(EMB_FILE):
    z = np.load(EMB_FILE)
    emb = dict(zip(z["ids"], z["vecs"]))
todo_ids = [r for r in scope_ids if r not in emb]
if todo_ids:
    texts = [f"{by_id.at[r, 'title']}. {by_id.at[r, 'abstract']}"[:8000] for r in todo_ids]   # <= 8192 tokens per input
    batches = [texts[i:i + 128] for i in range(0, len(texts), 128)]
    print(f"embedding {len(texts)} papers in {len(batches)} batches ...")
    res = pmap(lambda b: client.embed(EMBED, b), batches)
    emb.update(zip(todo_ids, [d["embedding"] for r in res for d in sorted(r["data"], key=lambda d: d["index"])]))
    np.savez(EMB_FILE, ids=np.array(list(emb)), vecs=np.array(list(emb.values()), dtype=np.float32))
E_ids = scope_ids
E = np.array([emb[r] for r in E_ids], dtype=np.float32)
E /= np.linalg.norm(E, axis=1, keepdims=True)
pos = {r: i for i, r in enumerate(E_ids)}
print(f"{len(E_ids)} jet papers embedded")

# nearest neighbours of the seeds, per task
cands = {}
for task_id, task in TASKS.items():
    seeds = [r for r in ref_df.loc[ref_df["task_id"] == task_id, "recid"] if r in pos]
    if seeds:
        S, seed_names = E[[pos[r] for r in seeds]], seeds
    else:   # no hand-reviewed paper in the pool: fall back to the task sentence
        S = np.array(client.embed(EMBED, [task["short_query"]])["data"][0]["embedding"], dtype=np.float32)[None]
        S, seed_names = S / np.linalg.norm(S), ["short_query"]
        print(f"! {task_id}: no seed paper in the pool -> using short_query")
    sims = E @ S.T                               # similarity of every paper to every seed
    best, closest = sims.max(axis=1), sims.argmax(axis=1)
    near = [E_ids[i] for i in np.argsort(-best) if E_ids[i] not in seeds][:N_NEIGHBOURS]
    picked = seeds + near
    # 4. reranker score (for the queue order in section 6)
    rr = dict(zip(picked, rerank_scores(task["short_query"], [docs[r] for r in picked])))
    rows = [{"recid": r, "similarity": 1.0, "closest_seed": r, "rerank_score": rr[r],
             "retrieval_method": "seed (hand-reviewed, notebook 03)"} for r in seeds]
    rows += [{"recid": r, "similarity": float(best[pos[r]]), "closest_seed": seed_names[closest[pos[r]]],
              "rerank_score": rr[r],
              "retrieval_method": f"seed-and-expand ({EMBED} cosine {best[pos[r]]:.2f} to {seed_names[closest[pos[r]]]})"}
             for r in near]
    df = pd.DataFrame(rows)
    df.insert(0, "task_id", task_id)
    df["title"] = df["recid"].map(by_id["title"])
    cands[task_id] = df
    df.to_csv(out_path(f"candidates_{task_id}.csv"), index=False)
    print(f"{task_id}: {len(seeds)} seeds + {len(near)} similar papers "
          f"(similarity {df['similarity'].iloc[len(seeds):].max():.2f} .. {df['similarity'].iloc[len(seeds):].min():.2f})")


embedding 3197 papers in 25 batches ...
3197 jet papers embedded
jet_pt_background_correction: 1 seeds + 100 similar papers (similarity 0.89 .. 0.70)
quenched_vs_unquenched_jet_classification: 4 seeds + 100 similar papers (similarity 0.97 .. 0.75)
jet_energy_loss_prediction: 2 seeds + 100 similar papers (similarity 0.86 .. 0.76)
jet_particle_vs_background_classification: 1 seeds + 100 similar papers (similarity 0.74 .. 0.68)


In [10]:
cands["quenched_vs_unquenched_jet_classification"][["recid", "similarity", "closest_seed", "rerank_score", "title"]].head(30)


,recid,similarity,closest_seed,rerank_score,title
0,3148534,1.000000,3148534,0.436308,Jet quenching identification via supervised learning in simulated heavy-ion collisions
1,3083458,1.000000,3083458,0.786956,Quantifying vacuum-like jets in heavy-ion collisions: a machine learning study
2,2854166,1.000000,2854166,0.695310,Model-Agnostic Tagging of Quenched Jets in Heavy-Ion Collisions
3,2091324,1.000000,2091324,0.575575,Identifying quenched jets in heavy ion collisions with machine learning
4,2162452,0.970054,2091324,0.377893,Study of jet Quenching in Quark Gluon Plasma With Machine Learning
5,2651429,0.862860,2091324,0.375102,Jet substructure observables for jet quenching in quark gluon plasma: A machine learning driven analysis
6,3080476,0.855895,2091324,0.645476,Validating a machine learning approach to identify quenched jets in heavy-ion collisions
7,2689747,0.838607,3148534,0.844158,Overview: Jet quenching with machine learning
8,2760065,0.838607,3148534,0.844224,Overview: Jet quenching with machine learning
9,1978762,0.822841,2091324,0.734049,The information content of jet quenching and machine learning assisted observable design


## 5. LLM screening → review queue
**Goal:** sort the candidates into candidate / uncertain / excluded from the abstract, and keep 20–30 per task for
verification.

Each paper is screened with **two different prompts**:
- **A** uses 4 labels: performs_task, mentions_only (the task is only used, cited or planned), different_task
  (absent, or the same words for another task), and uncertain.
- **B** first extracts the paper's input and target, then decides.

Alos So if A and B disagree, the paper considered as  uncertain.

In [12]:
RULES = """You screen physics papers for a catalogue of candidate downstream benchmark tasks for foundation models (FMs).
The task is defined by its scientific field and prediction objective. Papers with different inputs, detectors or architectures can perform the same task. Check which object receives a prediction and what that prediction means.
decision = "performs_task": this study evaluates an ML model on the specified target in heavy-ion physics and reports a numerical performance result. Record its actual input and output.
decision = "mentions_only": the target appears as motivation, previous work, a future plan or a standard measurement, without evaluation of an ML model on it.
decision = "different_task": the scientific setting, predicted object or output differs. For example, predicting a jet's energy loss differs from classifying that jet as quenched.
decision = "uncertain": the text is insufficient to establish the target or evaluation.

List any additional prediction objectives separately. Copy evidence_text as a contiguous excerpt from the supplied text."""

SCREEN_A = """Candidate task ({task_id}): {description}
Inputs seen in earlier studies (examples only): {input_examples}
Prediction target: {observable_or_target}

Title: {title}
Abstract: \"\"\"{abstract}\"\"\"

Use only the abstract. First state the paper's main contribution in one sentence, then decide.
Return one JSON object:
{{"main_contribution": "<one sentence, your words>",
  "decision": "performs_task"|"mentions_only"|"different_task"|"uncertain",
  "decision_reason": "<one sentence>",
  "evidence_text": "<verbatim contiguous quote from the abstract>"}}"""

SCREEN_B_SYSTEM = """Check the study's own ML evaluation. Identify its model input, prediction target and scientific field. The same target in heavy-ion physics can be predicted using different inputs or methods. Answer "performs_task" for a model evaluated on that target with a numerical performance result. Answer "uncertain" if the abstract leaves its output or evaluation unclear. Return only JSON."""

SCREEN_B = """Candidate task ({task_id}): {description}
Inputs seen in earlier studies (examples only): {input_examples}
Prediction target: {observable_or_target}

Title: {title}
Abstract: \"\"\"{abstract}\"\"\"

Return one JSON object:
{{"paper_input": "<what the paper's method or study takes as input, or unknown>",
  "paper_target": "<what it predicts, reconstructs or measures, or unknown>",
  "decision": "performs_task"|"does_not_perform"|"uncertain",
  "decision_reason": "<one sentence>"}}"""

TO_HANDOUT = {"performs_task": "candidate", "uncertain": "uncertain", "mentions_only": "excluded",
              "different_task": "excluded", "does_not_perform": "excluded"}

In [ ]:
def screen_one(job):
    task_id, rid = job
    fmt = dict(task_id=task_id, title=by_id.at[rid, "title"], abstract=by_id.at[rid, "abstract"], **TASKS[task_id])
    return chat_json(RULES, SCREEN_A.format(**fmt)), chat_json(SCREEN_B_SYSTEM, SCREEN_B.format(**fmt))


jobs = [(t, r) for t, df in cands.items() for r in df["recid"]]
print(f"screening {len(jobs)} task-paper pairs x 2 prompts with {WORKERS} workers ...")
screen_res = dict(zip(jobs, pmap(screen_one, jobs)))

queues = {}
for task_id, df in cands.items():
    df = df.copy()
    a = [screen_res[(task_id, r)][0] for r in df["recid"]]
    b = [screen_res[(task_id, r)][1] for r in df["recid"]]
    df["screen_a"] = [x.get("decision", "invalid") for x in a]
    df["screen_b"] = [x.get("decision", "invalid") for x in b]
    ha = df["screen_a"].map(TO_HANDOUT).fillna("uncertain")
    hb = df["screen_b"].map(TO_HANDOUT).fillna("uncertain")
    df["screen_decision"] = np.where(ha == hb, ha, "uncertain")      # split vote -> uncertain
    df["split_vote"] = ha != hb
    df["screen_reason"] = [x.get("decision_reason", "") for x in a]
    df.to_csv(out_path(f"screened_{task_id}.csv"), index=False)
    # queue: both prompts say performs_task > uncertain > excluded; then by reranker
    df["_p"] = df["screen_decision"].map({"candidate": 0, "uncertain": 1, "excluded": 2})
    df = df.sort_values(["_p", "rerank_score"], ascending=[True, False])
    q = df[df["_p"] < 2].head(QUEUE_MAX)
    if len(q) < QUEUE_MIN:
        q = pd.concat([q, df[df["_p"] == 2].head(QUEUE_MIN - len(q))])
    q = q.drop(columns="_p").reset_index(drop=True)
    q.insert(1, "queue_rank", range(1, len(q) + 1))
    queues[task_id] = q
    q.to_csv(out_path(f"queue_{task_id}.csv"), index=False)
    print(f"{task_id}: {df['screen_decision'].value_counts().to_dict()}, split votes {int(df['split_vote'].sum())} "
          f"-> queue {len(q)}")

screening 408 task-paper pairs x 2 prompts with 4 workers ...
jet_pt_background_correction: {'excluded': 60, 'candidate': 21, 'uncertain': 20}, split votes 20 -> queue 30
quenched_vs_unquenched_jet_classification: {'excluded': 87, 'candidate': 13, 'uncertain': 4}, split votes 4 -> queue 20
jet_energy_loss_prediction: {'excluded': 96, 'candidate': 4, 'uncertain': 2}, split votes 2 -> queue 20
jet_particle_vs_background_classification: {'excluded': 98, 'uncertain': 2, 'candidate': 1}, split votes 2 -> queue 20


In [14]:
# How did the screening judge your hand-reviewed papers? (they should come out as candidate or uncertain)
def screened_as(row):
    df = queues.get(row["task_id"])
    if df is None:
        return "-"
    full = pd.read_csv(out_path(f"screened_{row['task_id']}.csv"), dtype={"recid": str})
    m = full[full["recid"] == row["recid"]]
    return f"{m['screen_decision'].iloc[0]} (A: {m['screen_a'].iloc[0]}, B: {m['screen_b'].iloc[0]})" if len(m) else "not a candidate"


ref_df["screened_as"] = ref_df.apply(screened_as, axis=1)
ref_df[["task_id", "paper_id", "title", "candidate_rank", "screened_as"]]

,task_id,paper_id,title,candidate_rank,screened_as
0,jet_pt_background_correction,INSPIRE:1698396,Machine Learning based jet momentum reconstruction in heavy-ion collisions,1,"candidate (A: performs_task, B: performs_task)"
1,quenched_vs_unquenched_jet_classification,INSPIRE:3148534,Jet quenching identification via supervised learning in simulated heavy-ion collisions,1,"candidate (A: performs_task, B: performs_task)"
2,quenched_vs_unquenched_jet_classification,INSPIRE:3083458,Quantifying vacuum-like jets in heavy-ion collisions: a machine learning study,2,"candidate (A: performs_task, B: performs_task)"
3,quenched_vs_unquenched_jet_classification,INSPIRE:2854166,Model-Agnostic Tagging of Quenched Jets in Heavy-Ion Collisions,3,"candidate (A: performs_task, B: performs_task)"
4,quenched_vs_unquenched_jet_classification,INSPIRE:2091324,Identifying quenched jets in heavy ion collisions with machine learning,4,"candidate (A: performs_task, B: performs_task)"
5,jet_energy_loss_prediction,INSPIRE:3080476,Validating a machine learning approach to identify quenched jets in heavy-ion collisions,1,"candidate (A: performs_task, B: performs_task)"
6,jet_energy_loss_prediction,INSPIRE:1836454,Deep learning jet modifications in heavy-ion collisions,2,"candidate (A: performs_task, B: performs_task)"
7,jet_particle_vs_background_classification,INSPIRE:1854216,Jet characterization in Heavy Ion Collisions by QCD-Aware Graph Neural Networks,1,"candidate (A: performs_task, B: performs_task)"


## 7. Fetch the full texts of the queue papers
**Goal:** get the full text of the 20–30 queue papers per task.

only the queue papers are saved to fulltext_queue.jsonl; later runs fetch only papers that are new to the queue.

In [15]:
USE_FULLTEXT = True
FT_FILE = out_path("fulltext_queue.jsonl")
SKIP_HEAD = re.compile(r"acknowledg|^references?$|bibliograph|list of (tables|figures)|^contents$|author contrib", re.I)
RECID_RX = re.compile(r'^\{"recid": ?"?(\d+)')

fulltext = {}
if os.path.exists(FT_FILE):
    fulltext = {r["recid"]: r for r in map(json.loads, open(FT_FILE))}
need = {r for q in queues.values() for r in q["recid"]} - set(fulltext)
if USE_FULLTEXT and need:
    decades = sorted({by_id.at[r, "year"][:3] + "0s" for r in need})
    with open(FT_FILE, "a", encoding="utf-8") as out:
        for dec in decades:
            path = os.path.join(SRC, dec, f"fulltext_{dec}.jsonl.zst")
            print("scanning", path, "for", len(need), "papers ...")
            proc = subprocess.Popen(["zstd", "-dc", path], stdout=subprocess.PIPE, text=True, encoding="utf-8")
            for line in proc.stdout:
                m = RECID_RX.match(line)
                if not m or m.group(1) not in need:
                    continue
                r = json.loads(line)
                sections = [{"heading": strip_latex(s.get("heading") or ""), "text": strip_latex(s.get("text") or "")}
                            for s in r.get("sections") or []]
                rec = {"recid": m.group(1), "arxiv_id": r.get("arxiv_id") if r.get("arxiv_id") not in (None, "None", "") else "",
                       "sections": [s for s in sections if s["text"]]}
                out.write(json.dumps(rec, ensure_ascii=False) + "\n")
                fulltext[rec["recid"]] = rec
                need.discard(rec["recid"])
                if not need:
                    proc.kill()
                    break
            proc.wait()
queue_ids = {r for q in queues.values() for r in q["recid"]}
print(f"full text available for {len(queue_ids & set(fulltext))} of {len(queue_ids)} queue papers")

scanning /lustre/scratch/data/jdearrud_hpc-inspire/release_20261104/2010s/fulltext_2010s.jsonl.zst for 75 papers ...
scanning /lustre/scratch/data/jdearrud_hpc-inspire/release_20261104/2020s/fulltext_2020s.jsonl.zst for 48 papers ...
full text available for 68 of 75 queue papers


## 8. verification with evidence
**Goal:** decide every queue paper again, now with full-text passages, and get a quote as evidence.

- Passages: the full text is cut into ~1 500-character chunks, and the reranker picks the 3 that fit the task best.
  These go to the LLM together with the abstract. 
- Quote check: the quote must be found in the paper (quote_match). Its real location gives evidence_location;
  the location the LLM names is not trusted.
- **Decision:**

  | Verification says | Quote found? | Abstract screening agrees? | decision |
  |---|---|---|---|
  | performs_task | yes | yes | supported |
  | performs_task | no | yes | candidate |
  | mentions_only / different_task | – | yes | excluded |
  | anything | – | no | uncertain (a human decides) |

In [16]:
VERIFY_USER = """Candidate task ({task_id}): {description}
Inputs seen in earlier studies (examples only): {input_examples}
Prediction target: {observable_or_target}

Title: {title}
Abstract: \"\"\"{abstract}\"\"\"

{body_label}:
{body}

URLs found anywhere in the paper (with the text before them):
{urls}

Use only the text above. Never guess: if the text does not say it, answer "unknown" / "not_mentioned"; URLs must be copied exactly from the URL list. Give the metric and numerical ML performance for the specified target. If those are unclear, mark the match uncertain.
First state the paper's main contribution in one sentence, then decide.
Return one JSON object:
{{"main_contribution": "<one sentence, your words>",
  "decision": "performs_task"|"mentions_only"|"different_task"|"uncertain",
  "match_rationale": "<how the study evaluates its prediction of the specified heavy-ion target>",
  "study_type": "experimental"|"phenomenological"|"methodological"|"unspecified",
  "evidence_text": "<verbatim contiguous quote from the text above>",
  "paper_scope": "<the model input context supported by this study, or unknown>",
  "paper_input": "<the input this paper uses for the task, in its own words, or unknown>",
  "paper_target": "<the target it predicts, or unknown>",
  "metric": "<metric used to evaluate the task, or unknown>",
  "reported_result": "<headline result with numbers, or unknown>",
  "result_quote": "<verbatim contiguous quote from the text above that contains that result, or empty>",
  "evaluation_setting": "<data or simulation and conditions of the evaluation (generator, collision system, energy, centrality), or unknown>",
  "dataset": "<data or simulation sample the task is evaluated on (e.g. JEWEL, PYTHIA, ALICE Pb-Pb 5.02 TeV), or unknown>",
  "dataset_availability": "public_mentioned"|"restricted"|"not_mentioned",
  "dataset_url": "<one URL from the URL list that gives access to that data, or empty>",
  "code": "<name of the software or model implementing the task, or unknown>",
  "code_availability": "public_mentioned"|"restricted"|"not_mentioned",
  "code_url": "<one URL from the URL list for that code, or empty>",
  "other_tasks": ["<short label of any OTHER distinct task the study itself carries out and reports a result for; not future work, not sub-steps of this task>"]}}"""

URL_RX = re.compile(r"https?://[^\s<>\"'\)\]\}]+")
AVAIL = {"public_mentioned": "mentioned_unverified", "restricted": "restricted", "not_mentioned": "not_checked"}


def chunks(sections):
    out = []
    for s in sections:
        if SKIP_HEAD.search(s["heading"]):
            continue
        sents = re.split(r"(?<=[.!?])\s+", s["text"])
        sents = [x[i:i + CHUNK] for x in sents for i in range(0, len(x), CHUNK)]   # tables have no full stops
        buf = ""
        for sent in sents:
            if len(buf) + len(sent) > CHUNK and buf:
                out.append((s["heading"], buf.strip()))
                buf = ""
            buf += sent + " "
        if buf.strip():
            out.append((s["heading"], buf.strip()))
    return out


def top_passages(query, sections, k=PASSAGES):
    ch = chunks(sections)
    if not ch:
        return []
    sc = [client.rerank(RERANK, query, [f"{h}. {t}" for h, t in ch[i:i + 200]], top_n=len(ch[i:i + 200]))["results"]
          for i in range(0, len(ch), 200)]
    scores = [0.0] * len(ch)
    for b, res in enumerate(sc):
        for x in res:
            scores[b * 200 + x["index"]] = x["relevance_score"]
    best = sorted(range(len(ch)), key=lambda i: -scores[i])[:k]
    return [ch[i] for i in sorted(best)]                 # keep document order


def locate(quote, rid):
    """Where the quote really is: 'abstract', 'title', 'section: <heading>' or '' (not found)."""
    p = by_id.loc[rid]
    secs = (fulltext.get(rid) or {}).get("sections") or []
    for name, txt in [("abstract", p["abstract"]), ("title", p["title"])] + \
                     [(f"section: {s['heading'] or '(no heading)'}", s["text"]) for s in secs]:
        m = quote_match(quote, txt)
        if m in ("exact", "normalized", "fragments"):
            return ("abstract" if name.lower() == "section: abstract" else name), m
    return "", quote_match(quote, p["abstract"])

In [17]:
def verify_one(job):
    task_id, rid = job
    task, p = TASKS[task_id], by_id.loc[rid]
    secs = (fulltext.get(rid) or {}).get("sections") or [] if USE_FULLTEXT else []
    try:
        # task passages + one passage about data/code availability (both picked by the free reranker)
        ps = top_passages(task["short_query"], secs)
        ps += [x for x in top_passages("public data release, simulation sample, code repository, software availability "
                                       "(GitHub, Zenodo, HEPData)", secs, 1) if x not in ps]
    except Exception as e:                                  # rerank failed: verify on the abstract alone
        FAILED_CALLS.append(f"rerank {rid}: {type(e).__name__}: {str(e)[:200]}")
        ps = []
    body, label = (("\n\n".join(f"## {h}\n{t}" for h, t in ps), f"Full-text passages (the {len(ps)} most relevant)")
                   if ps else ("(no full text available)", "Full text"))
    full = " ".join(s["text"] for s in secs)
    urls = sorted({u.rstrip(".,;:") for u in URL_RX.findall(full)})[:40]
    url_lines = "\n".join(f"- {u}  ({full[max(0, full.find(u) - 120):max(0, full.find(u))].strip()[-120:]})"
                            for u in urls) or "(none)"
    a = chat_json(RULES, VERIFY_USER.format(task_id=task_id, title=p["title"], abstract=p["abstract"], urls=url_lines,
                                            body=body, body_label=label, **task), max_tokens=2500)
    loc, m = locate(a.get("evidence_text") or "", rid)
    ok_url = lambda u: u.rstrip(".,;:") if u and u.rstrip(".,;:") in urls else ""     # noqa: E731
    rq = a.get("result_quote") or ""
    res_ok = bool(rq) and quote_match(rq, p["abstract"] + "\n" + full) in ("exact", "normalized", "fragments")
    result = a.get("reported_result") or "unknown"
    return {**a, "evidence_location": loc, "quote_match": m, "has_fulltext": bool(ps),
            "dataset_url": ok_url(a.get("dataset_url")), "code_url": ok_url(a.get("code_url")),
            "url_rejected": "; ".join(u for u in (a.get("dataset_url"), a.get("code_url")) if u and not ok_url(u)),
            "reported_result": result if (res_ok or result == "unknown") else f"[quote not found] {result}"}


jobs = [(t, r) for t, q in queues.items() for r in q["recid"]]
print(f"verifying {len(jobs)} queue papers ...")
ver_res = dict(zip(jobs, pmap(verify_one, jobs)))

rows = []
for task_id, q in queues.items():
    task = TASKS[task_id]
    for _, r in q.iterrows():
        v, p = ver_res[(task_id, r["recid"])], by_id.loc[r["recid"]]
        d = v.get("decision", "invalid")
        found = v["quote_match"] in ("exact", "normalized", "fragments") and bool(v["evidence_location"])
        decision = ("supported" if found else "candidate") if d == "performs_task" else TO_HANDOUT.get(d, "uncertain")
        disagree = (r["screen_decision"] == "candidate") != (d == "performs_task")
        if disagree:
            decision = "uncertain"
        arxiv = (fulltext.get(r["recid"]) or {}).get("arxiv_id") or ""
        rows.append({
            "field": FIELD, "fm_scope": v.get("paper_scope") or "unknown", "task_id": task_id, "task_label": task["task_label"],
            "task_type": task["task_type"], "observable_or_target": v.get("paper_target") or "unknown",
            "input_modality": v.get("paper_input") or "unknown",
            "study_type": v.get("study_type", "unspecified"),
            "paper_id": f"arXiv:{arxiv}" if arxiv else f"INSPIRE:{r['recid']}",
            "paper_url": f"https://arxiv.org/abs/{arxiv}" if arxiv else f"https://inspirehep.net/literature/{r['recid']}",
            "title": p["title"], "year": p["year"], "study_group_id": "",
            "match_rationale": v.get("match_rationale", ""),
            "evidence_text": v.get("evidence_text", "") if found else "",
            "evidence_location": v["evidence_location"], "decision": decision, "review_status": "automated_proposal",
            "retrieval_method": f"{r['retrieval_method']}; screening {MODEL} 2 prompts {PROMPT_VERSION} (split vote -> uncertain); "
                                f"verification {MODEL} abstract + {PASSAGES} passages ({RERANK})",
            "dataset_status": AVAIL.get(v.get("dataset_availability"), "not_checked"), "dataset_url": v["dataset_url"],
            "model_code_status": AVAIL.get(v.get("code_availability"), "not_checked"), "model_code_url": v["code_url"],
            "metric": v.get("metric") or "unknown", "reported_result": v["reported_result"],
            "evaluation_setting": v.get("evaluation_setting") or "unknown",
            "dataset": v.get("dataset") or "unknown", "model_code_name": v.get("code") or "unknown",
            "url_rejected": v["url_rejected"],
            # diagnostics
            "recid": r["recid"], "queue_rank": r["queue_rank"], "screen_decision": r["screen_decision"],
            "verify_decision": d, "screen_verify_disagree": disagree,
            "exclusion_reason": d if decision == "excluded" else "", "quote_match": v["quote_match"],
            "unverified_quote": "" if found else v.get("evidence_text", ""), "has_fulltext": v["has_fulltext"],
            "other_tasks": "; ".join(x for x in v.get("other_tasks") or [] if isinstance(x, str)),
        })
cat = pd.DataFrame(rows)
print("failed API calls so far (their papers are treated as uncertain):", len(FAILED_CALLS))
cat.groupby("task_id")["decision"].value_counts().unstack(fill_value=0)

verifying 90 queue papers ...
! HTTP 500 on /rerank, retry 1/4 in 1s
failed API calls so far (their papers are treated as uncertain): 0


decision,candidate,excluded,supported,uncertain
task_id,,,,
jet_energy_loss_prediction,0,16,4,0
jet_particle_vs_background_classification,0,18,1,1
jet_pt_background_correction,1,1,20,8
quenched_vs_unquenched_jet_classification,0,4,12,4


In [ ]:
# How did screening and verification judge your hand-reviewed papers
def verified_as(row):
    m = cat[(cat["task_id"] == row["task_id"]) & (cat["recid"] == row["recid"])]
    if not len(m):
        return "not in queue"
    return f"{m['decision'].iloc[0]} (verify: {m['verify_decision'].iloc[0]}, quote: {m['quote_match'].iloc[0]})"


ref_df["verified_as"] = ref_df.apply(verified_as, axis=1)
ref_df[["task_id", "paper_id", "title", "candidate_rank", "screened_as", "verified_as"]]

,task_id,paper_id,title,candidate_rank,screened_as,verified_as
0,jet_pt_background_correction,INSPIRE:1698396,Machine Learning based jet momentum reconstruction in heavy-ion collisions,1,"candidate (A: performs_task, B: performs_task)","supported (verify: performs_task, quote: exact)"
1,quenched_vs_unquenched_jet_classification,INSPIRE:3148534,Jet quenching identification via supervised learning in simulated heavy-ion collisions,1,"candidate (A: performs_task, B: performs_task)","supported (verify: performs_task, quote: exact)"
2,quenched_vs_unquenched_jet_classification,INSPIRE:3083458,Quantifying vacuum-like jets in heavy-ion collisions: a machine learning study,2,"candidate (A: performs_task, B: performs_task)","supported (verify: performs_task, quote: exact)"
3,quenched_vs_unquenched_jet_classification,INSPIRE:2854166,Model-Agnostic Tagging of Quenched Jets in Heavy-Ion Collisions,3,"candidate (A: performs_task, B: performs_task)","supported (verify: performs_task, quote: exact)"
4,quenched_vs_unquenched_jet_classification,INSPIRE:2091324,Identifying quenched jets in heavy ion collisions with machine learning,4,"candidate (A: performs_task, B: performs_task)","supported (verify: performs_task, quote: exact)"
5,jet_energy_loss_prediction,INSPIRE:3080476,Validating a machine learning approach to identify quenched jets in heavy-ion collisions,1,"candidate (A: performs_task, B: performs_task)","supported (verify: performs_task, quote: exact)"
6,jet_energy_loss_prediction,INSPIRE:1836454,Deep learning jet modifications in heavy-ion collisions,2,"candidate (A: performs_task, B: performs_task)","supported (verify: performs_task, quote: exact)"
7,jet_particle_vs_background_classification,INSPIRE:1854216,Jet characterization in Heavy Ion Collisions by QCD-Aware Graph Neural Networks,1,"candidate (A: performs_task, B: performs_task)","supported (verify: performs_task, quote: exact)"


## 9. Duplicates (study_group_id) and your hand-reviewed rows
**Goal:** group duplicate versions of one study (thesis and paper, paper and proceedings), and put your notebook-03
decisions in.

- **Duplicates:** title + abstract embeddings (bge-m3) with cosine ≥ 0.95 are treated as the same study, and only the
  best-decided version keeps its decision. Pairs between 0.90 and 0.95 are listed in possible_duplicate_of for you to
  check.

In [ ]:
added = 0
for _, h in ref_df.iterrows():
    if h["task_id"] not in TASKS:
        continue
    m = (cat["task_id"] == h["task_id"]) & (cat["recid"] == h["recid"])
    keep = {k: h[k] for k in ["fm_scope", "observable_or_target", "input_modality", "study_type", "match_rationale",
                               "evidence_text", "evidence_location", "decision", "review_status"] if k in h}
    if m.any():
        for k, val in keep.items():
            cat.loc[m, k] = val
    else:
        p = by_id.loc[h["recid"]] if h["recid"] in by_id.index else {"title": h["title"], "year": h.get("year", "")}
        cat = pd.concat([cat, pd.DataFrame([{
            "field": FIELD, "task_id": h["task_id"], "task_label": TASKS[h["task_id"]]["task_label"],
            "task_type": TASKS[h["task_id"]]["task_type"], "paper_id": h["paper_id"], "paper_url": h["paper_url"],
            "title": p["title"], "year": p["year"], "study_group_id": "", "retrieval_method": "manual (notebook 03)",
            "recid": h["recid"], "queue_rank": 999, **keep}])],
            ignore_index=True)
        added += 1
print(f"hand-reviewed rows applied: {len(ref_df[ref_df['task_id'].isin(TASKS)])} ({added} were not in the verification queue and were added)")

hand-reviewed rows applied: 8 (0 were not in the verification queue and were added)


In [20]:
# study_group_id: union of papers whose title+abstract embeddings have cosine >= SAME_STUDY.
ids = sorted(set(cat["recid"]))
texts = [f"{by_id.at[r, 'title']}. {by_id.at[r, 'abstract']}"[:6000] if r in by_id.index else r for r in ids]
vecs = []
for i in range(0, len(texts), 64):
    vecs += [d["embedding"] for d in sorted(client.embed(EMBED, texts[i:i + 64])["data"], key=lambda d: d["index"])]
v = np.array(vecs)
v /= np.linalg.norm(v, axis=1, keepdims=True)
sim = v @ v.T

parent = {r: r for r in ids}


def root(x):
    while parent[x] != x:
        x = parent[x]
    return x


near = {}
for i in range(len(ids)):
    for j in range(i + 1, len(ids)):
        if sim[i, j] >= SAME_STUDY:
            a, b = sorted([root(ids[i]), root(ids[j])])
            parent[b] = a
        elif sim[i, j] >= CHECK_STUDY:
            near.setdefault(ids[i], []).append(f"INSPIRE:{ids[j]} ({sim[i, j]:.2f})")
            near.setdefault(ids[j], []).append(f"INSPIRE:{ids[i]} ({sim[i, j]:.2f})")
cat["study_group_id"] = cat["recid"].map(lambda r: f"sg_{root(r)}")
cat["possible_duplicate_of"] = cat["recid"].map(lambda r: "; ".join(near.get(r, [])))

# one row per task and study: the best-decided version (human-reviewed first) keeps its decision
order = {"supported": 0, "candidate": 1, "uncertain": 2, "excluded": 3}
cat["_o"] = cat["decision"].map(order) - 10 * (cat["review_status"] == "human_reviewed")
cat = cat.sort_values(["task_id", "_o", "queue_rank"])
dup = cat.duplicated(["task_id", "study_group_id"])
first_id = cat.groupby(["task_id", "study_group_id"])["paper_id"].transform("first")
cat.loc[dup, "exclusion_reason"] = "duplicate version of " + first_id[dup]
cat.loc[dup, "decision"] = "excluded"
cat = cat.drop(columns="_o").sort_values(["task_id", "queue_rank"]).reset_index(drop=True)
print("duplicate versions folded:", int(dup.sum()), "| pairs to check by hand:", int((cat["possible_duplicate_of"] != "").sum()))

duplicate versions folded: 6 | pairs to check by hand: 0


In [ ]:
# Save every paper  with all its details as JSON, grouped by task.
final = cat[~cat["exclusion_reason"].astype(str).str.startswith("duplicate version")]
# final = final[final["decision"] != "excluded"]   # <- uncomment to keep only supported / candidate / uncertain

# Only these fields go into the JSON 
KEEP = ["field", "fm_scope", "task_id", "task_label", "task_type", "observable_or_target", "input_modality",
        "study_type", "paper_id", "paper_url", "study_group_id", "match_rationale", "evidence_text",
        "evidence_location", "decision", "review_status", "retrieval_method",
        "metric", "reported_result", "evaluation_setting"]

# layout {"tasks": [{"task_id": ..., "papers": [rows]}]}: the one validate.py and the scorer read
tasks = []
for task_id, g in final.groupby("task_id"):
    tasks.append({
        "task_id": task_id,
        "task_label": TASKS[task_id]["task_label"],
        "n_papers": len(g),
        "decisions": g["decision"].value_counts().to_dict(),
        "papers": g[KEEP].fillna("").to_dict(orient="records"),
    })
out = {"tasks": tasks}

JSON_FILE = out_path("papers_after_duplicates.json")
with open(JSON_FILE, "w", encoding="utf-8") as f:
    json.dump(out, f, indent=1, ensure_ascii=False, default=str)
print("saved", os.path.abspath(JSON_FILE), "|", len(final), "papers,", (cat.shape[0] - len(final)), "duplicates removed")


saved /home/s28rraoo_hpc/hackaton/api_outputs_notebook_04_seed_only/papers_after_duplicates.json | 84 papers, 6 duplicates removed
